# 用 LangChain 串接提示詞：讓本機 Phi-3 一步步寫出「標題 → 主角 → 故事」
---

## 這堂課你會學到什麼

1. 為什麼要「串接」提示詞：把大任務拆成幾個小任務，後一步使用前一步的結果
2. LangChain 的三個基本零件：**PromptTemplate**（模板）、**LLM**（模型）、**`|`**（串接）
3. `prompt | llm` 的輸出只有「一個字串」，所以不能直接接到下一個需要多個變數的模板
4. **`RunnablePassthrough.assign(...)`**：保留前面的資料，並新增一個欄位，這是本課的核心
5. 為什麼每次呼叫都會「重跑前面的步驟」，以及如何讓結果穩定（`temperature = 0`）

## 0. 環境準備

```bash
pip install langchain-community langchain-core llama-cpp-python
```

- `llama-cpp-python` 要用 GPU 加速，需要依平台用支援 CUDA 的方式安裝；只裝一般版本也能跑，但是 CPU 模式，比較慢。
- 需要先把 `Phi-3-mini-4k-instruct-q4.gguf`（約 2.39 GB）下載到本機，並在第 3 節的 `MODEL_PATH` 填上正確路徑。
- 版本備註：`langchain-community` 與 `langchain-core` 的細節會隨版本變動。我實際檢查原始碼的版本是 `langchain-community 0.4.2`、`langchain-core 1.6.12`。

## 1. 先看整體：為什麼要「串接」？

要請模型寫一個故事，可以一次把所有要求塞進一個提示詞，但小模型常常顧不過來。更穩的做法是**拆成三個小任務**，每個任務只做一件事：

```
 summary ──► [任務1：寫標題] ──► title
 summary + title ──► [任務2：寫主角] ──► character
 summary + title + character ──► [任務3：寫故事] ──► story
```

後一步會用到前一步的結果，這就是**提示詞串接（prompt chaining）**。

LangChain 的角色，是幫你管理「模板」、「呼叫模型」、「把結果交給下一步」這幾件事。

### 這堂課出現的名詞

| 名詞 | 意思 |
|---|---|
| `PromptTemplate` | 含有 `{變數}` 的提示詞模板，之後再填入內容 |
| `LlamaCpp` | LangChain 提供的包裝，讓它可以呼叫本機 llama.cpp 載入的 GGUF 模型 |
| Runnable | LangChain 裡「可以被 `invoke` 執行」的物件的統稱：模板、模型、鏈都是 |
| `\|` | 把兩個 Runnable 接起來，前一個的輸出當後一個的輸入 |
| LCEL | LangChain Expression Language，就是這種用 `\|` 組合的寫法 |
| `invoke` | 執行一個 Runnable，傳入輸入、取得輸出 |

## 2. 匯入套件

| 匯入 | 用途 |
|---|---|
| `LlamaCpp` | 載入本機的 GGUF 模型 |
| `PromptTemplate` | 建立提示詞模板 |
| `RunnablePassthrough` | 本課的重點，用它的 `assign` 來「保留資料並新增欄位」 |

In [1]:
from langchain_community.llms import LlamaCpp
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

/tmp/ipykernel_94517/3332923397.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.llms import LlamaCpp


## 3. 設定模型檔路徑

`MODEL_PATH` 是你電腦上 GGUF 檔案的位置。`"../"` 代表「上一層資料夾」。請依照你實際存放的位置修改。

In [2]:
MODEL_PATH = "../Phi-3-mini-4k-instruct-q4.gguf"

## 4. 三個提示詞模板

### Phi-3 的聊天格式

Phi-3 是用特定格式訓練的，提示詞要長這樣：

```
<s><|user|>
（使用者說的話）
<|end|>
<|assistant|>
```

模型會接在 `<|assistant|>` 後面寫它的回答，寫完之後會輸出 `<|end|>`。我們在載入模型時設定了 `stop = ["<|end|>"]`，所以看到它就停止。

### 模板中的 `{變數}`

| 模板 | 需要的變數 |
|---|---|
| `TITLE_TEMPLATE` | `{summary}` |
| `CHARACTER_TEMPLATE` | `{summary}`、`{title}` |
| `STORY_TEMPLATE` | `{summary}`、`{title}`、`{character}` |

**注意變數越來越多：** 第二個模板用到第一步的產出 `title`，第三個模板用到前兩步的產出 `title` 和 `character`。這正是後面要用 `assign` 的原因。

### ⚠️ 三引號字串裡的空白會算進提示詞

`"""..."""` 裡面的縮排與換行，會原封不動變成提示詞的一部分。所以模板內容要**靠左寫**（程式裡的註解也這樣提醒）。如果模板每行前面多了空格，這些空格會被送給模型，可能影響輸出。

In [3]:
#包含多個提示詞的鍊 生成一個故事包括 標題 主角敘述 故事摘要
# 注意：三引號字串裡的縮排、換行都會變成提示詞的一部分，所以模板內容靠左寫。
TITLE_TEMPLATE = """
<s><|user|>
Create a title of a story about {summary}. Only return the title.
<|end|>
<|assistant|>
"""
CHARACTER_TEMPLATE = """
<s><|user|>
Describe the main character of a story about {summary} with the title {title}.
Use only two sentences.
<|end|>
<|assistant|>
"""

STORY_TEMPLATE= """
<s><|user|>
Create a story about {summary} with the title {title}. The main character is: {character}.
Only return the story and it cannot be longer than one paragraph.
<|end|>
<|assistant|>
"""

## 5. 載入模型

```python
llm = LlamaCpp(model_path = MODEL_PATH, n_gpu_layers = -1, max_tokens = 500, n_ctx = 2048,
               seed = 42, verbose = False, stop = ["<|end|>"], temperature = 0)
```

| 參數 | 意思 |
|---|---|
| `model_path` | GGUF 檔案的位置 |
| `n_gpu_layers = -1` | 要放到 GPU 的層數；`-1` = 全部都放 GPU |
| `max_tokens = 500` | 每次最多生成 500 個 token（再長就被截斷） |
| `n_ctx = 2048` | 上下文長度：提示詞 + 生成內容加起來最多 2048 個 token |
| `seed = 42` | 亂數種子 |
| `verbose = False` | 不顯示 llama.cpp 的詳細日誌；出錯時改成 `True` |
| `stop = ["<|end|>"]` | 看到這個字串就停止生成 |
| `temperature = 0` | **不隨機**：每一步都選機率最高的字，結果穩定 |

### 為什麼要加 `temperature = 0`？

`langchain-community 0.4.2` 的原始碼，`LlamaCpp` 的預設值是：

| 參數 | 預設 |
|---|---|
| `temperature` | 0.8 |
| `top_p` | 0.95 |
| `top_k` | 40 |
| `repeat_penalty` | 1.1 |

`temperature = 0.8` 是有隨機性的。你的程式每個 `ask_*` 都會**重新跑一遍**，如果有隨機性，三次的標題就會不一樣。設成 0 之後，同樣的提示詞通常會得到同樣的結果，方便比較與除錯。

In [4]:
llm = LlamaCpp(
    model_path = MODEL_PATH,
    n_gpu_layers = -1,
    max_tokens = 500,
    n_ctx = 2048,
    seed = 42,
    verbose = False,
    stop = ["<|end|>"],
    temperature = 0
)

## 6. 建立提示詞模板物件

`PromptTemplate.from_template(...)` 會自動從字串中找出 `{...}` 的變數名稱，不用自己再列。

### 如何看到「填好變數之後的完整提示詞」？

直接用模板的 `format`。這是在送進模型**之前**就組好的，所以看到的就是模型實際收到的內容。

In [5]:
title_prompt = PromptTemplate.from_template(template=TITLE_TEMPLATE)
character_prompt = PromptTemplate.from_template(template=CHARACTER_TEMPLATE)
story_prompt = PromptTemplate.from_template(template=STORY_TEMPLATE)

## 7. 第一個零件：`prompt | llm`

```python
title_prompt | llm
```

`|` 的意思是「前一個的輸出，當成後一個的輸入」：

```
{"summary": "..."}  ──►  title_prompt  ──►  填好的提示詞  ──►  llm  ──►  "模型寫的標題"（字串）
```

### 兩個要點

1. **`|` 其實就是 `RunnableSequence`。** LangChain 在背後把 `a | b | c` 轉成 `RunnableSequence(a, b, c)`。下面的實驗會證明。
2. **輸出只有「一個字串」。** `prompt | llm` 跑完，原本輸入的 `summary` 就不見了，只剩模型的回答。

這就帶出本課最重要的問題：**第二個模板需要 `summary` 和 `title` 兩個變數，但上一步只留下一個字串，怎麼辦？**

## 8. 核心：`RunnablePassthrough.assign(...)`

### 一句話

> **把收到的 dict 原樣留著，再多加一個欄位，欄位的內容是某個小任務的結果。**

可以想成**在表格後面加一欄**。

```python
RunnablePassthrough.assign(title = title_prompt | llm)
```

```
輸入：  {"summary": "a girl..."}
              │
              ├─ 原樣保留 ───────────────────────┐
              │                                  ▼
              └─ 跑小任務（填模板 + 模型）──► "Echoes of..."
                                                 │
輸出：  {"summary": "a girl...", "title": "Echoes of..."}
```

- 括號裡等號左邊的名字（`title`），就是新欄位的名字。
- **這個名字要和下一個模板裡的 `{變數}` 同名。** 下一個模板寫 `{title}`，這裡就要叫 `title`。
- 新欄位的名字，相當於舊版 `LLMChain` 的 `output_key`。

## 9. 建立三條 chain，再接成完整流程

| chain | 收到的輸入 | 小任務 | 新增的欄位 | 輸出 |
|---|---|---|---|---|
| `title_chain` | `{summary}` | 標題模板 + 模型 | `title` | `{summary, title}` |
| `character_chain` | `{summary, title}` | 主角模板 + 模型 | `character` | `{summary, title, character}` |
| `story_chain` | `{summary, title, character}` | 故事模板 + 模型 | `story` | `{summary, title, character, story}` |

三條各自做一件事，最後用 `|` 接起來：

```python
llm_chain = title_chain | character_chain | story_chain
```

每一段都拿到「上一段的整包 dict」，所以：
- 主角模板可以用 `{title}`（第一段新增的）
- 故事模板可以用 `{title}` 和 `{character}`（前兩段新增的）

### 欄位名稱要對得上

```
title_chain      assign(title = ...)       ──►  下一個模板寫 {title}
character_chain  assign(character = ...)   ──►  下一個模板寫 {character}
```

In [9]:
# RunnablePassthrough.assign(名字 = 小任務)：
#   把輸入的 dict 原樣保留，再多加一個欄位「名字」，內容是這個小任務的輸出。
#   欄位名字就相當於舊版 LLMChain 的 output_key。
title_chain = RunnablePassthrough.assign(title = title_prompt | llm)
character_chain = RunnablePassthrough.assign(character = character_prompt | llm)
story_chain = RunnablePassthrough.assign(story = story_prompt | llm)

#把這些小任務接成完整工作流程
llm_chain = title_chain | character_chain | story_chain

> 📌 **本節重點**
> - `|` 負責**接起來**；`assign` 負責**不弄丟前面的資料，並把這一步的答案存成新欄位**。
> - 新欄位的名字必須和下一個模板的 `{變數}` 同名。
> - `character_chain` 不能單獨使用，因為它需要 `title`，要接在 `title_chain` 後面（或由你自己在輸入裡提供）。

## 10. 設定要輸入的主題

整條流程只需要一個輸入：`summary`（故事的主題）。

> 小提醒：`summary` 結尾的句點會跟著填進模板。標題模板是 `about {summary}. Only return the title.`，所以會變成 "...her mother.. Only return the title."（兩個句點）；角色與故事模板則會變成 "...her mother. with the title..." 這種句子中間有句點的樣子（上面第 6 節的實驗已經印出標題那一個）。可以試著把結尾的 `.` 拿掉，比較結果有沒有不同。

In [ ]:
summary = "a girl that lost her mother"

## 11. 只跑到標題

```python
title_chain.invoke({"summary": summary})
```

回傳的是 dict：`{'summary': ..., 'title': ...}`。取標題要用 `result["title"]`。

> 模型輸出的字串，開頭常有空白。顯示時可以用 `.strip()` 去掉。

In [12]:
# 回傳 dict：{'summary': ..., 'title': ...}
title_response = title_chain.invoke({"summary" : summary})
print(title_response)

{'summary': 'a girl that lost her mother.', 'title': ' "Whispers of a Mother\'s Love: A Tale of Loss and Healing"\n\n(Note: The title is crafted to evoke emotion while hinting at the storyline involving loss, motherly love, and healing.)'}


## 12. 跑到主角

```python
(title_chain | character_chain).invoke({"summary": summary})
```

這裡會**自動先生成標題**，再用標題與主題生成主角。回傳 `{'summary', 'title', 'character'}`。

### ⚠️ 重複呼叫模型

這一格會**重新生成標題**，而不是沿用上一格的結果。所以：

| 呼叫 | 模型被呼叫的次數 |
|---|---|
| 標題（第 11 節） | 1 次 |
| 主角（本節） | 2 次（標題 + 主角） |
| 故事（第 13 節） | 3 次（標題 + 主角 + 故事） |
| 合計 | **6 次** |

在本機跑小模型，這會花時間。如果只要最後的故事，直接跑第 13 節就好，它的結果已經包含 `title` 與 `character`。

而且如果有隨機性（`temperature` 大於 0），三次的標題會不一樣；設成 0 之後，同樣的提示詞通常會得到同樣的標題。

In [13]:
# 回傳 dict：{'summary': ..., 'title': ..., 'character': ...}
character_response = (title_chain | character_chain).invoke({"summary" : summary})
print(character_response)

{'summary': 'a girl that lost her mother.', 'title': ' "Whispers of a Mother\'s Love: A Tale of Loss and Healing"\n\n(Note: The title is crafted to evoke emotion while hinting at the storyline involving loss, motherly love, and healing.)', 'character': ' Eliza, the protagonist in "Whispers of a Mother\'s Love: A Tale of Loss and Healing," is an introspective teenager grappling with profound grief after her mother\'s untimely passing. Her journey through sorrow reveals a resilient spirit that finds solace in the cherished memories and enduring love shared by her mother, ultimately guiding Eliza towards healing and self-discovery.'}


## 13. 跑完整流程

```python
llm_chain.invoke({"summary": summary})
```

回傳 `{'summary', 'title', 'character', 'story'}`。

In [14]:
# 回傳 dict：{'summary', 'title', 'character', 'story'}
story_response = llm_chain.invoke({"summary" : summary})
print(story_response)

{'summary': 'a girl that lost her mother.', 'title': ' "Whispers of a Mother\'s Love: A Tale of Loss and Healing"\n\n(Note: The title is crafted to evoke emotion while hinting at the storyline involving loss, motherly love, and healing.)', 'character': ' Eliza, the protagonist in "Whispers of a Mother\'s Love: A Tale of Loss and Healing," is an introspective teenager grappling with profound grief after her mother\'s untimely passing. Her journey through sorrow reveals a resilient spirit that finds solace in the cherished memories and enduring love shared by her mother, ultimately guiding Eliza towards healing and self-discovery.', 'story': ' In the heart of a small, sun-dappled town lived Eliza, whose world was irrevocably altered when her mother\'s whispers became silent. "Whispers of a Mother\'s Love: A Tale of Loss and Healing" unfolds as Eliza navigates through an ocean of grief, each wave carrying the weight of memories that anchor her to her mother\'s tender embrace. Through quie

## 15. 全課重點整理

### 流程回顧

| 步驟 | 程式 | 作用 |
|---|---|---|
| ① 準備模板 | `TITLE_TEMPLATE` 等三個字串 | 含 `{變數}` 的 Phi-3 格式提示詞 |
| ② 載入模型 | `LlamaCpp(...)` | 在本機載入 GGUF；`temperature = 0` 讓結果穩定 |
| ③ 建立模板物件 | `PromptTemplate.from_template(...)` | 自動找出 `{變數}` |
| ④ 每一段 | `RunnablePassthrough.assign(名字 = 模板 \| llm)` | 保留前面資料，並新增一個欄位 |
| ⑤ 串起來 | `title_chain \| character_chain \| story_chain` | 完整流程 |
| ⑥ 執行 | `.invoke({"summary": ...})` | 傳入 dict，取得 dict |

### 必記的 7 件事

1. **串接提示詞** = 把大任務拆成小任務，後一步用前一步的結果。
2. `prompt | llm` 的輸出只有**一個字串**，輸入的欄位會不見。
3. **`|` 就是 `RunnableSequence`**，只是簡寫。
4. **`RunnablePassthrough.assign(名字 = 小任務)`**：保留輸入 dict，再新增一個欄位。
5. 新欄位的名字要和**下一個模板的 `{變數}`** 同名，名字不一致就會缺變數。
6. 每個 `ask_*` 都從頭重跑：標題 1 次、主角 2 次、故事 3 次，合計 6 次呼叫。
7. `temperature = 0` 讓結果穩定；**不要加 `echo = True`**（會把提示詞一起帶到下一步）。

### 常見錯誤

| 現象 | 可能原因與處理 |
|---|---|
| `TypeError: Expected mapping type as input to PromptTemplate` | 把字串接到需要 dict 的模板；改用 `assign` |
| 缺少變數的錯誤（如缺 `title`） | 單獨使用 `character_chain`，或 `assign` 的名字和模板變數對不上 |
| 三次標題都不一樣 | 預設 `temperature = 0.8` 有隨機性；改成 0 |
| 標題裡混進一大段提示詞 | `echo = True` 造成；請拿掉 |
| 輸出開頭有空白 | 顯示前 `.strip()` |
| 模型不聽話，寫了額外說明 | 小模型的限制；把提示詞寫更明確，或調整 `max_tokens` |
| 載入模型失敗 | 檢查 `MODEL_PATH`；`verbose=True` 看詳細原因；顯存不夠時降低 `n_gpu_layers` 或 `n_ctx` |

### 動手練習

1. 把 `summary` 結尾的句點拿掉，比較結果的差別。
2. 新增第四段：請模型為故事寫一句 slogan。要做幾件事？（提示：新的模板、新的 `assign`、接在 `story_chain` 後面，欄位名字要對上模板裡的變數。）
3. 把 `temperature` 改成 `0.8`，連續跑兩次 `title_chain`，觀察標題是否不同。
4. 把 `character_chain` 單獨拿來 `invoke({"summary": summary})`，觀察錯誤訊息，並說明為什麼會出錯。
5. 把 `llm_chain` 改成明確的 `RunnableSequence(title_chain, character_chain, story_chain)`，確認結果相同。
6. 進階：在 `assign` 的小任務後面加上去除空白，讓 `title` 不再有開頭空白。（提示：可以在 `| llm` 後面再接一個 `RunnableLambda(lambda s: s.strip())`。）

### 延伸方向

- `StrOutputParser`：把模型輸出明確轉成字串，換成聊天模型（如 `ChatOpenAI`）時會需要
- 舊版 `LLMChain` 與 `SequentialChain`：已移到 `langchain_classic`，概念與 `assign` 相同
- 讓步驟之間能「平行」進行（`RunnableParallel`）
- 把同一個流程換成 OpenAI 的模型，只需要換掉 `llm`